# 99 — Results for Thesis

Aggregates all saved metrics into clean tables and figures ready for the thesis.

**Depends on:** `01_preprocess_text.ipynb` + completed training & evaluation runs.

No models are loaded or training is done here — everything is read from JSON files.

In [ ]:
import os
import sys

_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'src')) else os.path.abspath(os.path.join(_here, '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import json
import pandas as pd
import matplotlib.pyplot as plt

from src.config import EXP_ROOT, VERSION, MODELS_TO_RUN
from src.training.utils import slugify

## Load saved metrics

In [ ]:
def load_json(path: str) -> dict:
    with open(path, encoding='utf-8') as f:
        return json.load(f)

records = []
histories = {}

for model_name in MODELS_TO_RUN:
    slug    = slugify(model_name)
    run_dir = os.path.join(EXP_ROOT, VERSION, slug)

    train_m = load_json(os.path.join(run_dir, 'train_metrics.json'))
    eval_m  = load_json(os.path.join(run_dir, 'eval_metrics.json'))

    records.append({
        'model':             model_name,
        'total_params':      train_m['params']['total_params'],
        'trainable_params':  train_m['params']['trainable_params'],
        'best_epoch':        train_m['best_epoch'],
        'best_dev_macro_f1': train_m['best_dev_macro_f1'],
        'train_time_s':      train_m['train_time_s'],
        'avg_power_w':       train_m['power'].get('avg_power_w', float('nan')),
        'energy_kwh':        train_m['power'].get('energy_kwh',  float('nan')),
        'test_accuracy':     eval_m['accuracy'],
        'test_macro_f1':     eval_m['macro_f1'],
        'test_weighted_f1':  eval_m['weighted_f1'],
        'infer_ms_bs1':      eval_m['inference_ms_per_sample_bs1'],
    })

    history_path = os.path.join(run_dir, 'history.csv')
    if os.path.exists(history_path):
        histories[model_name] = pd.read_csv(history_path)

results_df = pd.DataFrame(records)
results_df

## Performance table

In [ ]:
perf = results_df[[
    'model', 'test_accuracy', 'test_macro_f1', 'test_weighted_f1', 'infer_ms_bs1'
]].copy()
perf.columns = ['Model', 'Accuracy', 'Macro F1', 'Weighted F1', 'Infer ms (bs=1)']
perf = perf.set_index('Model').round(4)
perf

## Efficiency table

In [ ]:
eff = results_df[[
    'model', 'total_params', 'train_time_s', 'avg_power_w', 'energy_kwh'
]].copy()
eff['total_params_M'] = eff['total_params'] / 1e6
eff['train_time_min'] = eff['train_time_s'] / 60
eff = eff[['model', 'total_params_M', 'train_time_min', 'avg_power_w', 'energy_kwh']]
eff.columns = ['Model', 'Params (M)', 'Train time (min)', 'Avg power (W)', 'Energy (kWh)']
eff = eff.set_index('Model').round(3)
eff

## Learning curves

In [ ]:
fig, axes = plt.subplots(1, len(histories), figsize=(7 * len(histories), 4), squeeze=False)

for ax, (model_name, hist) in zip(axes[0], histories.items()):
    ax.plot(hist['epoch'], hist['train_loss'],   label='Train loss',   marker='o')
    ax.plot(hist['epoch'], hist['dev_loss'],     label='Dev loss',     marker='s')
    ax.plot(hist['epoch'], hist['dev_macro_f1'], label='Dev macro-F1', marker='^')
    ax.set_xlabel('Epoch')
    ax.set_title(model_name)
    ax.legend()

plt.suptitle('Learning curves', fontsize=13)
plt.tight_layout()
plt.show()

## Macro F1 bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    results_df['model'].str.replace('-', '\n'),
    results_df['test_macro_f1'],
    color=['steelblue', 'darkorange'][:len(results_df)],
    edgecolor='white',
)
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_ylim(0, 1)
ax.set_ylabel('Macro F1 (test)')
ax.set_title('Model comparison — Macro F1')
plt.tight_layout()
plt.show()